In [4]:
import pandas as pd 

In [61]:
customers=pd.read_csv("..\\data\\raw\\olist_customers_dataset.csv")
Geolocation=pd.read_csv("..\\data\\raw\\olist_geolocation_dataset.csv")
CustomerOrderDetails=pd.read_csv("..\\data\\raw\\olist_orders_dataset.csv")
ProductPaymentDetails=pd.read_csv("..\\data\\raw\\olist_order_payments_dataset.csv")
OrderReview=pd.read_csv("..\\data\\raw\\olist_order_reviews_dataset.csv")
ProductCategory=pd.read_csv("..\\data\\raw\\product_category_name_translation.csv")
ProductDetails=pd.read_csv("..\\data\\raw\\olist_products_dataset.csv")
SellerDetails=pd.read_csv("..\\data\\raw\\olist_sellers_dataset.csv")
OrderItems=pd.read_csv("..\\data\\raw\\olist_order_items_dataset.csv")

<h2>Preprocessing Customer Order details</h2>

In [5]:
CustomerOrderDetails.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


<h3>Checking NULL values</h3>

In [6]:
CustomerOrderDetails.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [7]:
CustomerOrderDetails.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


<h3>Converting date column to actual date

In [8]:
CustomerOrderDetails["order_purchase_timestamp"] = CustomerOrderDetails["order_purchase_timestamp"].apply(
    pd.to_datetime
)

In [9]:
date_columns = [
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

CustomerOrderDetails[date_columns] = CustomerOrderDetails[date_columns].apply(
    pd.to_datetime
)

In [10]:
CustomerOrderDetails.dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

<h3>Create cleaned CSV file</h3>

In [34]:
CustomerOrderDetails.to_csv('..\\data\\cleaned\\olist_orders_dataset_cleaned.csv', index=False)

<h2>Preprocessing Geolocation</h2>

<h3>Handling duplicate records</h2>

In [64]:
Geolocation[Geolocation["geolocation_zip_code_prefix"].duplicated()]

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
2,1046,-23.546129,-46.642951,sao paulo,SP
10,1013,-23.547325,-46.634184,sao paulo,SP
13,1012,-23.548946,-46.634671,sao paulo,SP
14,1037,-23.545187,-46.637855,são paulo,SP
15,1046,-23.546081,-46.644820,sao paulo,SP
...,...,...,...,...,...
1000158,99950,-28.068639,-52.010705,tapejara,RS
1000159,99900,-27.877125,-52.224882,getulio vargas,RS
1000160,99950,-28.071855,-52.014716,tapejara,RS
1000161,99980,-28.388932,-51.846871,david canabarro,RS


In [67]:
Geolocation["geolocation_zip_code_prefix"].duplicated().sum()

np.int64(0)

In [66]:
Geolocation = Geolocation.drop_duplicates(subset=["geolocation_zip_code_prefix"], keep="first")

<h3>Create cleaned CSV file</h3>

In [68]:
Geolocation.to_csv('..\\data\\cleaned\\olist_geolocation_dataset_cleaned.csv', index=False)

In [69]:
Geolocation.shape

(19015, 5)

<h2>PreprocessingCustomer dataset</h2>

<h3>Validating foreign key reference column</h3>

In [30]:
mismatched_customers = customers[
    ~customers['customer_zip_code_prefix'].isin(
        Geolocation['geolocation_zip_code_prefix']
    )
]
mismatched_customers

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
354,ecb1725b26e8b8c458181455dfa434ea,b55a113bb84fc10eaf58c6d09ec69794,72300,brasilia,DF
382,bcf86029aeed4ed8bac0e16eb14c22f5,7cd7974c9f79f75b77f323878ef87f43,11547,cubatao,SP
877,f4302056f0c58570522590f8181de2c7,67b05b597a66b5c449025000b9430abb,64605,picos,PI
1218,03bbe0ce5c28e05f22917607db798818,8f3dca4306d5a89e4ae2c65c110603a2,72465,brasilia,DF
1272,ad4950aded55c2ea376be59506456d68,aa2b96dd03307ea6dc4b763c0b5f0b39,7729,caieiras,SP
...,...,...,...,...,...
97467,cf818420383856a129134f5f8343f7b8,795c495a65f983b242fb01bd507977c5,72338,brasilia,DF
97780,67f3e907dce402e696b15f9308ff22ed,6f232f2f5c7f33b7bd9d794d2afacadd,68629,paragominas,PA
98140,f792e419335df11d82c32efcfb09c51b,c04c085b8e7573ba87b9ae1968d0985e,28530,sao sebastiao do paraiba,RJ
98878,78a11bb1fa72f556996b9a5b9bcd0629,e7536f62a200b415edd9491ac12a17fa,55863,siriji,PE


<h3>Handling invalid data</h3>

In [31]:
customers['customer_zip_code_prefix'] = customers['customer_zip_code_prefix'].astype('Int64')

mismatched_customers = ~customers['customer_zip_code_prefix'].isin(
    Geolocation['geolocation_zip_code_prefix']
)

customers.loc[mismatched_customers, 'customer_zip_code_prefix'] = pd.NA

In [32]:
customers['customer_zip_code_prefix'].isnull().sum()

np.int64(278)

In [35]:
customers.shape

(99441, 5)

<h3>Create cleaned CSV file</h3>

In [34]:
customers.to_csv('..\\data\\cleaned\\olist_customers_dataset_cleaned.csv', index=False)

<h2>Preprocessing Customer Order Reviews</h2>

In [57]:
OrderReview.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


<h3>Checking NULL values</h3>

In [16]:
OrderReview.isnull().sum()

review_id                  0
order_id                   0
review_score               0
review_comment_title       0
review_comment_message     0
review_creation_date       0
review_answer_timestamp    0
dtype: int64

<h3>Filling NULL values</h3>

In [58]:
OrderReview[['review_comment_title', 'review_comment_message']] = OrderReview[['review_comment_title', 'review_comment_message']].fillna(' ')

In [59]:
OrderReview.isnull().sum()

review_id                  0
order_id                   0
review_score               0
review_comment_title       0
review_comment_message     0
review_creation_date       0
review_answer_timestamp    0
dtype: int64

<h3>Validating duplicate values</h3>

In [40]:
OrderReview["review_id"].duplicated().sum()

np.int64(814)

In [50]:
OrderReview[OrderReview["review_id"].duplicated()]

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
3317,3242cc306a9218d0377831e175d62fbf,9c5bfba7de6a4abbb6ba0baab78d1622,5,NaN,Ótimo,2017-07-21 00:00:00,2017-07-22 01:06:37
5719,308316408775d1600dad81bd3184556d,3fe4dbcdb046a475dbf25463c1ca78bd,5,NaN,Ajudem a rastrear uma das 3 prateleiras que fa...,2017-09-07 00:00:00,2017-09-11 09:58:09
7213,8ee90ac383cf825bb7f4756130d4e74a,75d5d3d16567a27eefc5752aeb063072,5,NaN,recomendo,2017-05-30 00:00:00,2017-06-02 18:16:17
7805,d433c252647c51309432ca0b763f969b,191abde9c3bf77d1ce67d100d8db6d15,1,NaN,NaN,2017-09-03 00:00:00,2017-09-06 16:11:00
9013,03a6a25db577d0689440933055111897,3fde8b7313af6b37b84b5c7594d7add0,5,NaN,Muito Bom! Gostei Bastante! Tecido Ótimo! Aten...,2017-12-15 00:00:00,2017-12-16 01:32:18
...,...,...,...,...,...,...,...
99108,2c6c08892b83ba4c1be33037c2842294,42ae1967f68c90bb325783ac55d761ce,4,NaN,"Chegou um pouco amassada, mas nada de mais, e ...",2017-07-03 00:00:00,2017-07-05 19:06:59
99124,6ec93e77f444e0b1703740a69122e35d,e1fdc6e9d1ca132377e862593a7c0bd4,5,NaN,Vendedor compromisso do vou o cliente,2017-10-07 00:00:00,2017-10-07 19:47:11
99164,2afe63a67dfd99b3038f568fb47ee761,c5334d330e36d2a810a7a13c72e135ee,5,NaN,"Muito bom, produto conforme anunciado, entrega...",2018-03-03 00:00:00,2018-03-04 22:56:47
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30


In [38]:
OrderReview[OrderReview[["review_id","order_id"]].duplicated()]

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp


<h3>Converting date column to actual date</h3>

In [60]:
date_columns = [
    "review_creation_date",
    "review_answer_timestamp"
]

OrderReview[date_columns] = OrderReview[date_columns].apply(
    pd.to_datetime
)

In [13]:
OrderReview.dtypes

review_id                             str
order_id                              str
review_score                        int64
review_comment_title                  str
review_comment_message                str
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object

<h3>Create cleaned CSV file

In [61]:
OrderReview.to_csv('..\\data\\cleaned\\olist_order_reviews_dataset_cleaned.csv', index=False)

<h2>Preprocessing Product Details</h2>

<h3>Check NULL values</h3>

In [51]:
ProductDetails=pd.read_csv("..\\data\\raw\\olist_products_dataset.csv")


In [52]:
ProductDetails.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [53]:
ProductDetails.isnull().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

<h3>Handling invalid records</h3>

In [ ]:
ProductDetails[ProductDetails[["product_category_name","product_name_lenght","product_description_lenght","product_photos_qty","product_weight_g","product_length_cm","product_height_cm","product_width_cm"]].isnull().all(axis=1)]

In [64]:
ProductDetails.dropna(
    subset=[
        "product_category_name",
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ],
    how="all",
    inplace=True
)

In [74]:
ProductDetails[~ProductDetails['product_category_name'].isin(
    ProductCategory['product_category_name']
)]

,product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,0.0,0.0,0.0,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,0.0,0.0,0.0,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,0.0,0.0,0.0,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,0.0,0.0,0.0,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,0.0,0.0,0.0,300.0,35.0,7.0,12.0
...,...,...,...,...,...,...,...,...,...
32515,b0a0c5dd78e644373b199380612c350a,NaN,0.0,0.0,0.0,1800.0,30.0,20.0,70.0
32589,10dbe0fbaa2c505123c17fdc34a63c56,NaN,0.0,0.0,0.0,800.0,30.0,10.0,23.0
32616,bd2ada37b58ae94cc838b9c0569fecd8,NaN,0.0,0.0,0.0,200.0,21.0,8.0,16.0
32772,fa51e914046aab32764c41356b9d4ea4,NaN,0.0,0.0,0.0,1300.0,45.0,16.0,45.0


In [75]:
mismatched_product = ~ProductDetails['product_category_name'].isin(
    ProductCategory['product_category_name']
)

ProductDetails.loc[mismatched_product, 'product_category_name'] = pd.NA

<h3>Filling NULL values</h3>

In [66]:
ProductDetails=ProductDetails.fillna({
    "product_name_lenght": 0,
    "product_description_lenght": 0,
    "product_photos_qty": 0,
    "product_weight_g":0,
    "product_length_cm":0,
    "product_height_cm":0,
    "product_width_cm":0
})

In [67]:
ProductDetails.isnull().sum()

product_id                      0
product_category_name         610
product_name_lenght             0
product_description_lenght      0
product_photos_qty              0
product_weight_g                0
product_length_cm               0
product_height_cm               0
product_width_cm                0
dtype: int64

<h3>Renaming Columns</h3>

In [68]:
ProductDetails=ProductDetails.rename(columns={"product_name_lenght": "product_name_length", "product_description_lenght": "product_description_length"})

In [69]:
ProductDetails.head()

,product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


<h3>Create cleaned CSV file</h3>

In [76]:
ProductDetails.to_csv('..\\data\\cleaned\\olist_products_dataset_cleaned.csv', index=False)

In [73]:
ProductDetails[ProductDetails[["product_category_name","product_name_length","product_description_length","product_photos_qty","product_weight_g","product_length_cm","product_height_cm","product_width_cm"]].isnull().all(axis=1)]

,product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm


<h2>Preprocessing Product Order Item</h2>

<h3>Checking NULL values</h3>

In [77]:
OrderItems.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  str    
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  str    
 3   seller_id            112650 non-null  str    
 4   shipping_limit_date  112650 non-null  str    
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 6.0 MB


<h3>Converting date column to actual date</h3>

In [53]:
OrderItems["shipping_limit_date"] = OrderItems["shipping_limit_date"].apply(pd.to_datetime)

<h3>Create cleaned CSV file</h3>

In [62]:
OrderItems.to_csv('..\\data\\cleaned\\olist_order_items_dataset_cleaned.csv', index=False)

<h2>Preprocessing Seller details</h2>

<h3>Handling invalid data</h3>

In [81]:
SellerDetails.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [79]:
SellerDetails['seller_zip_code_prefix'] = SellerDetails['seller_zip_code_prefix'].astype('Int64')

mismatched_customers = ~SellerDetails['seller_zip_code_prefix'].isin(
    Geolocation['geolocation_zip_code_prefix']
)

SellerDetails.loc[mismatched_customers, 'seller_zip_code_prefix'] = pd.NA

In [80]:
SellerDetails.to_csv('..\\data\\cleaned\\olist_sellers_dataset_cleaned.csv', index=False)